# 03 — Modular RAG

Modular RAG decompoe o pipeline em **componentes independentes e intercambiaveis**.
Cada componente tem uma interface padronizada e pode ser substituido sem afetar os outros.

## Principios

```
Interface padronizada para cada componente:

  Indexer   → index(documents) → None
  Retriever → retrieve(query, top_k) → List[Document]
  Reranker  → rerank(query, docs, top_k) → List[Document]
  Generator → generate(query, docs) → str
  Validator → validate(question, answer, docs) → bool
```

**Vantagens:**
- Testar componentes individualmente
- Substituir sem refatoracao
- Escalar partes independentemente
- A/B testing de componentes

**Inspirado em:** Haystack, LlamaIndex Pipelines

In [ ]:
import sys
sys.path.insert(0, '..')

from abc import ABC, abstractmethod
from dataclasses import dataclass, field
from typing import List, Optional
import numpy as np
import ollama
from sentence_transformers import SentenceTransformer, CrossEncoder
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams, PointStruct
from rank_bm25 import BM25Okapi
import re

@dataclass
class Document:
    id: str
    text: str
    score: float = 0.0
    metadata: dict = field(default_factory=dict)

print('Interfaces definidas!')

In [ ]:
# Interfaces abstratas (contratos)
class BaseRetriever(ABC):
    @abstractmethod
    def retrieve(self, query: str, top_k: int = 5) -> List[Document]:
        pass

class BaseReranker(ABC):
    @abstractmethod
    def rerank(self, query: str, docs: List[Document], top_k: int = 5) -> List[Document]:
        pass

class BaseGenerator(ABC):
    @abstractmethod
    def generate(self, query: str, docs: List[Document]) -> str:
        pass

# Implementacoes concretas

class DenseRetriever(BaseRetriever):
    def __init__(self, embedding_model, qdrant_client, collection_name):
        self.model = embedding_model
        self.client = qdrant_client
        self.collection = collection_name
    
    def retrieve(self, query: str, top_k: int = 5) -> List[Document]:
        q_vec = self.model.encode(query, normalize_embeddings=True)
        results = self.client.query_points(self.collection, query=q_vec.tolist(), limit=top_k, with_payload=True).points
        return [Document(id=str(r.id), text=r.payload.get('text', ''), score=r.score) for r in results]

class BM25Retriever(BaseRetriever):
    def __init__(self, corpus: List[str]):
        self.corpus = corpus
        tokenized = [re.sub(r'[^a-z0-9 ]', '', d.lower()).split() for d in corpus]
        self.bm25 = BM25Okapi(tokenized)
    
    def retrieve(self, query: str, top_k: int = 5) -> List[Document]:
        tokens = re.sub(r'[^a-z0-9 ]', '', query.lower()).split()
        scores = self.bm25.get_scores(tokens)
        top_idx = np.argsort(scores)[::-1][:top_k]
        return [Document(id=str(i), text=self.corpus[i], score=float(scores[i])) for i in top_idx]

class CrossEncoderReranker(BaseReranker):
    def __init__(self, model_name='cross-encoder/ms-marco-MiniLM-L-6-v2'):
        print(f'Carregando cross-encoder...')
        self.cross_encoder = CrossEncoder(model_name)
    
    def rerank(self, query: str, docs: List[Document], top_k: int = 5) -> List[Document]:
        if not docs:
            return docs
        pairs = [(query, d.text) for d in docs]
        scores = self.cross_encoder.predict(pairs)
        ranked = sorted(zip(scores, docs), key=lambda x: x[0], reverse=True)
        result = [d for _, d in ranked[:top_k]]
        for i, (score, _) in enumerate(ranked[:top_k]):
            result[i].score = float(score)
        return result

class IdentityReranker(BaseReranker):
    """Sem reranking — passa direto (para comparacao)."""
    def rerank(self, query: str, docs: List[Document], top_k: int = 5) -> List[Document]:
        return docs[:top_k]

class OllamaGenerator(BaseGenerator):
    def __init__(self, model_name='llama3.2'):
        self.model_name = model_name
    
    def generate(self, query: str, docs: List[Document]) -> str:
        context = '\n\n'.join(d.text for d in docs)
        prompt = f'Responda com base APENAS no contexto.\n\nContexto:\n{context}\n\nPergunta: {query}\n\nResposta:'
        try:
            response = ollama.chat(model=self.model_name, messages=[{'role': 'user', 'content': prompt}])
            return response['message']['content']
        except Exception as e:
            return f'[LLM error: {e}]'

print('Componentes definidos!')

In [ ]:
# Pipeline Modular
class ModularRAGPipeline:
    def __init__(
        self,
        retriever: BaseRetriever,
        reranker: BaseReranker,
        generator: BaseGenerator,
        top_k_retrieve: int = 20,
        top_k_final: int = 5,
        name: str = 'ModularRAG',
    ):
        self.retriever = retriever
        self.reranker = reranker
        self.generator = generator
        self.top_k_retrieve = top_k_retrieve
        self.top_k_final = top_k_final
        self.name = name
    
    def run(self, query: str, verbose: bool = False) -> dict:
        # 1. Retrieve
        docs = self.retriever.retrieve(query, top_k=self.top_k_retrieve)
        if verbose:
            print(f'[Retrieve] {len(docs)} docs (top score: {docs[0].score:.3f})')
        
        # 2. Rerank
        reranked = self.reranker.rerank(query, docs, top_k=self.top_k_final)
        if verbose:
            print(f'[Rerank] {len(reranked)} docs finais')
        
        # 3. Generate
        answer = self.generator.generate(query, reranked)
        
        return {'answer': answer, 'sources': reranked, 'pipeline': self.name}


# Setup
embed_model = SentenceTransformer('all-MiniLM-L6-v2')
qdrant = QdrantClient(host='localhost', port=6333)

# Reusar colecao do notebook anterior
COLLECTION = 'advanced_rag'  # criado no notebook 02

# Corpus para BM25 (pegar da colecao)
scroll_result, _ = qdrant.scroll(COLLECTION, limit=500, with_payload=True)
corpus_bm25 = [p.payload.get('text', '') for p in scroll_result]

# Construir pipelines com diferentes componentes
dense_retriever = DenseRetriever(embed_model, qdrant, COLLECTION)
bm25_retriever = BM25Retriever(corpus_bm25)
cross_reranker = CrossEncoderReranker()
identity_reranker = IdentityReranker()
generator = OllamaGenerator('llama3.2')

# Pipeline A: Dense + sem reranking
pipeline_a = ModularRAGPipeline(dense_retriever, identity_reranker, generator, name='Dense-NoRerank')

# Pipeline B: Dense + Cross-Encoder
pipeline_b = ModularRAGPipeline(dense_retriever, cross_reranker, generator, name='Dense+CrossEncoder')

print('3 pipelines criados!')

In [ ]:
# A/B test: comparar pipelines
import time

query = 'Quais sao as tecnicas de quantizacao para vetores de alta dimensao?'

for pipeline in [pipeline_a, pipeline_b]:
    t0 = time.time()
    result = pipeline.run(query, verbose=True)
    elapsed = time.time() - t0
    
    print(f'\n--- Pipeline: {result["pipeline"]} ---')
    print(f'Latencia: {elapsed:.2f}s')
    print(f'Top-1 source: {result["sources"][0].text[:100]}...')
    print(f'Resposta: {result["answer"][:300]}')

## Vantagens do Modular RAG

```
Producao:  DenseRetriever + CrossEncoderReranker + OllamaGenerator
           (qualidade maxima)

Baixa Latencia: DenseRetriever + IdentityReranker + OllamaGenerator
                (mais rapido)

Sem GPU:   BM25Retriever + IdentityReranker + OllamaGenerator
           (menor custo computacional)
```

**Principio chave:** Trocar um componente nao requer refatorar o pipeline inteiro.

## Proximo
- [04 — Agentic RAG](04_agentic_rag.html)